# Find out the maximum Consequitive delays:-


In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Departure Delay analysis").getOrCreate()
df_delays=spark.read.\
    option("inferSchema", "true").\
        option("header","true").\
            csv("/databricks-datasets/flights/departuredelays.csv")
df_delays.show()
df_delays.printSchema()

+-------+-----+--------+------+-----------+
|   date|delay|distance|origin|destination|
+-------+-----+--------+------+-----------+
|1011245|    6|     602|   ABE|        ATL|
|1020600|   -8|     369|   ABE|        DTW|
|1021245|   -2|     602|   ABE|        ATL|
|1020605|   -4|     602|   ABE|        ATL|
|1031245|   -4|     602|   ABE|        ATL|
|1030605|    0|     602|   ABE|        ATL|
|1041243|   10|     602|   ABE|        ATL|
|1040605|   28|     602|   ABE|        ATL|
|1051245|   88|     602|   ABE|        ATL|
|1050605|    9|     602|   ABE|        ATL|
|1061215|   -6|     602|   ABE|        ATL|
|1061725|   69|     602|   ABE|        ATL|
|1061230|    0|     369|   ABE|        DTW|
|1060625|   -3|     602|   ABE|        ATL|
|1070600|    0|     369|   ABE|        DTW|
|1071725|    0|     602|   ABE|        ATL|
|1071230|    0|     369|   ABE|        DTW|
|1070625|    0|     602|   ABE|        ATL|
|1071219|    0|     569|   ABE|        ORD|
|1080600|    0|     369|   ABE| 

In [0]:
#checking distinct combination
from pyspark.sql.functions import expr
df_delays.select("origin","destination").distinct().count()

4138

In [0]:
#checking number of delays which is more than 0
from pyspark.sql.functions import col
df_delays.filter(col("delay") > 0).count()


591727

In [0]:
#checking number of delays which is less than 0
df_delays.filter(col("delay") < 0).count()

668729

In [0]:
#now considering setting window spec
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number,lag
window_spec=Window.partitionBy("origin","destination").orderBy("date")




In [0]:
from pyspark.sql.functions import when
df_delays=df_delays.withColumn("prev_delay",lag("delay",1).over(window_spec)).\
    withColumn("group_id",when(((col("prev_delay") <= 0) & (col("delay") > 0)),1).\
        when(((col("prev_delay").isNull()) & (col("delay") > 0)),1).otherwise(0))
df_delays.show()

+-------+-----+--------+------+-----------+----------+--------+
|   date|delay|distance|origin|destination|prev_delay|group_id|
+-------+-----+--------+------+-----------+----------+--------+
|1010605|   -5|     494|   ABQ|        DFW|      NULL|       0|
|1010730|   -1|     494|   ABQ|        DFW|        -5|       0|
|1010835|   -7|     494|   ABQ|        DFW|        -1|       0|
|1011030|   -2|     494|   ABQ|        DFW|        -7|       0|
|1011135|   -2|     494|   ABQ|        DFW|        -2|       0|
|1011405|    1|     494|   ABQ|        DFW|        -2|       1|
|1011520|    0|     494|   ABQ|        DFW|         1|       0|
|1020605|  118|     494|   ABQ|        DFW|         0|       1|
|1020730|   87|     494|   ABQ|        DFW|       118|       0|
|1020835|   59|     494|   ABQ|        DFW|        87|       0|
|1021030|   85|     494|   ABQ|        DFW|        59|       0|
|1021135|  115|     494|   ABQ|        DFW|        85|       0|
|1021405|    8|     494|   ABQ|        D

In [0]:
#now setting the filling all 
window_spec_with_group=Window.partitionBy("origin","destination").orderBy("date")

In [0]:
df_delays.show()

+-------+-----+--------+------+-----------+----------+--------+
|   date|delay|distance|origin|destination|prev_delay|group_id|
+-------+-----+--------+------+-----------+----------+--------+
|1010605|   -5|     494|   ABQ|        DFW|      NULL|       0|
|1010730|   -1|     494|   ABQ|        DFW|        -5|       0|
|1010835|   -7|     494|   ABQ|        DFW|        -1|       0|
|1011030|   -2|     494|   ABQ|        DFW|        -7|       0|
|1011135|   -2|     494|   ABQ|        DFW|        -2|       0|
|1011405|    1|     494|   ABQ|        DFW|        -2|       1|
|1011520|    0|     494|   ABQ|        DFW|         1|       0|
|1020605|  118|     494|   ABQ|        DFW|         0|       1|
|1020730|   87|     494|   ABQ|        DFW|       118|       0|
|1020835|   59|     494|   ABQ|        DFW|        87|       0|
|1021030|   85|     494|   ABQ|        DFW|        59|       0|
|1021135|  115|     494|   ABQ|        DFW|        85|       0|
|1021405|    8|     494|   ABQ|        D

In [0]:
from pyspark.sql.functions import sum
df_delays.withColumn("group_number",when(col("delay")<= 0,0).otherwise(sum("group_id").over(window_spec))).orderBy("origin","destination","date").show()

+-------+-----+--------+------+-----------+----------+--------+------------+
|   date|delay|distance|origin|destination|prev_delay|group_id|group_number|
+-------+-----+--------+------+-----------+----------+--------+------------+
|1011245|    6|     602|   ABE|        ATL|      NULL|       1|           1|
|1020605|   -4|     602|   ABE|        ATL|         6|       0|           0|
|1021245|   -2|     602|   ABE|        ATL|        -4|       0|           0|
|1030605|    0|     602|   ABE|        ATL|        -2|       0|           0|
|1031245|   -4|     602|   ABE|        ATL|         0|       0|           0|
|1040605|   28|     602|   ABE|        ATL|        -4|       1|           2|
|1041243|   10|     602|   ABE|        ATL|        28|       0|           2|
|1050605|    9|     602|   ABE|        ATL|        10|       0|           2|
|1051245|   88|     602|   ABE|        ATL|         9|       0|           2|
|1060625|   -3|     602|   ABE|        ATL|        88|       0|           0|

In [0]:
df_delays=df_delays.withColumn("group_number",when(col("delay")<= 0,0).otherwise(sum("group_id").over(window_spec)))

In [0]:
from pyspark.sql.functions import desc
df_delays.filter(col("group_number")> 0).groupBy("origin","destination","group_number").count().orderBy(desc("count")).show(5)

+------+-----------+------------+-----+
|origin|destination|group_number|count|
+------+-----------+------------+-----+
|   BNA|        PHL|          33|   44|
|   HOU|        SAT|           1|   42|
|   MDW|        MHT|          20|   41|
|   LAX|        HOU|           2|   39|
|   PHL|        ATL|          12|   39|
+------+-----------+------------+-----+
only showing top 5 rows


In [0]:
df_delays.filter((col("origin")=='BNA')& (col("destination")=='PHL') &(col("group_number")==33)).orderBy("date").show()

+-------+-----+--------+------+-----------+----------+--------+------------+
|   date|delay|distance|origin|destination|prev_delay|group_id|group_number|
+-------+-----+--------+------+-----------+----------+--------+------------+
|3081800|    3|     587|   BNA|        PHL|        -2|       1|          33|
|3091205|    9|     587|   BNA|        PHL|         3|       0|          33|
|3091845|  106|     587|   BNA|        PHL|         9|       0|          33|
|3101205|    8|     587|   BNA|        PHL|       106|       0|          33|
|3101845|   27|     587|   BNA|        PHL|         8|       0|          33|
|3111205|  112|     587|   BNA|        PHL|        27|       0|          33|
|3111845|   21|     587|   BNA|        PHL|       112|       0|          33|
|3121205|   12|     587|   BNA|        PHL|        21|       0|          33|
|3121845|   84|     587|   BNA|        PHL|        12|       0|          33|
|3131205|   18|     587|   BNA|        PHL|        84|       0|          33|

In [0]:
df_delays.filter((col("origin")=='BNA')& (col("destination")=='PHL')).orderBy("date").show(10)

+-------+-----+--------+------+-----------+----------+--------+------------+
|   date|delay|distance|origin|destination|prev_delay|group_id|group_number|
+-------+-----+--------+------+-----------+----------+--------+------------+
|1011110|    0|     587|   BNA|        PHL|      NULL|       0|           0|
|1011950|   68|     587|   BNA|        PHL|         0|       1|           1|
|1021110|   40|     587|   BNA|        PHL|        68|       0|           1|
|1021950|    0|     587|   BNA|        PHL|        40|       0|           0|
|1031110|  177|     587|   BNA|        PHL|         0|       1|           2|
|1031950|  116|     587|   BNA|        PHL|       177|       0|           2|
|1041000|   12|     587|   BNA|        PHL|       116|       0|           2|
|1041820|   25|     587|   BNA|        PHL|        12|       0|           2|
|1051110|  103|     587|   BNA|        PHL|        25|       0|           2|
|1051950|  123|     587|   BNA|        PHL|       103|       0|           2|